# Retail Customer Segmentation and Campaign Response Prediction

Analyzing 600 retail customer records to identify meaningful customer segments, predict campaign response / high-value status, and produce business recommendations.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, ConfusionMatrixDisplay)

df = pd.read_csv('retail_customer_project_dataset.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'retail_customer_project_dataset.csv'

## Part A: Data Loading and Exploration

In [ ]:
print("===== First 10 rows =====")
display(df.head(10))

print("\n===== Shape =====")
print(df.shape)

print("\n===== Column names =====")
print(df.columns.tolist())

print("\n===== Data types =====")
print(df.dtypes)

print("\n===== Missing values =====")
print(df.isnull().sum())

In [ ]:
print("===== Summary statistics (numerical columns) =====")
display(df.describe())

print("\n===== Counts by Gender =====")
print(df['Gender'].value_counts())

print("\n===== Counts by CampaignResponse =====")
print(df['CampaignResponse'].value_counts())
print("(", df['CampaignResponse'].value_counts(normalize=True).round(3).to_dict(), ")")

print("\n===== Counts by HighValueCustomer =====")
print(df['HighValueCustomer'].value_counts())

**Note:** `CampaignResponse` is heavily imbalanced (~91% Yes / 9% No). This matters later -- a model that always predicts "Yes" would score ~91% accuracy without learning anything, so accuracy alone would be misleading for that target.

In [ ]:
# Outlier check via boxplots + IQR method
numeric_cols = ['Age', 'AnnualIncome', 'SpendingScore', 'WebsiteVisitsPerMonth',
                 'AvgSessionMinutes', 'PreviousPurchases', 'MembershipYears', 'PurchaseAmount']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    axes[i].boxplot(df[col].dropna())
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

print("===== Outlier counts (IQR method) =====")
for col in numeric_cols:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"{col}: {n_outliers} outliers (bounds: {lower:.1f} to {upper:.1f})")

**Observation on outliers:** `SpendingScore` shows 141 "outliers" by the strict IQR rule, but this is a distribution artifact, not a data quality issue: 459 of 600 customers (76.5%) sit exactly at the maximum score of 100 (a ceiling effect), which compresses the IQR to zero and flags nearly everything below 100 as an "outlier." The other columns show more typical, low outlier counts (single digits to ~20).

## Part B: Data Preprocessing

In [ ]:
# Handle missing values (median for numeric, mode for categorical) before scaling/encoding
df['AnnualIncome'] = df['AnnualIncome'].fillna(df['AnnualIncome'].median())
df['AvgSessionMinutes'] = df['AvgSessionMinutes'].fillna(df['AvgSessionMinutes'].median())
df['EmailOpened'] = df['EmailOpened'].fillna(df['EmailOpened'].mode()[0])
print("Missing values remaining:", df.isnull().sum().sum())

# Separate numerical and categorical columns
categorical_cols = ['Gender', 'DiscountUsed', 'EmailOpened', 'CampaignResponse', 'HighValueCustomer']
numerical_cols = ['Age', 'AnnualIncome', 'SpendingScore', 'WebsiteVisitsPerMonth',
                   'AvgSessionMinutes', 'PreviousPurchases', 'MembershipYears', 'PurchaseAmount']
print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

In [ ]:
# Encode categorical columns
df_encoded = df.copy()
for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col + '_encoded'] = le.fit_transform(df_encoded[col])
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Scale numerical features
scaler = StandardScaler()
scaled_array = scaler.fit_transform(df_encoded[numerical_cols])
df_scaled = pd.DataFrame(scaled_array, columns=[c + '_scaled' for c in numerical_cols])
print("\nScaled numerical features (first 5 rows):")
display(df_scaled.head())

**Why encoding and scaling are required before clustering and classification:**

- **Encoding:** Algorithms like K-Means and Logistic Regression operate on numbers, not text -- they compute distances or weighted sums. `"Yes"`/`"No"` or `"Male"`/`"Female"` have no numeric meaning until encoded, so encoding is what makes categorical information usable by the math underneath these models at all.
- **Scaling:** K-Means, KNN, PCA, and t-SNE all rely on distance calculations. `AnnualIncome` ranges into the tens of thousands while `MembershipYears` ranges 0-8 -- without scaling, income differences alone would dominate every distance calculation, effectively drowning out every other feature regardless of how relevant it actually is. `StandardScaler` puts every numeric feature on the same footing (mean 0, standard deviation 1) so no single feature's raw magnitude can hijack the result.

## Part C: Exploratory Data Analysis and Visualization

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, ['Age', 'AnnualIncome', 'SpendingScore', 'PurchaseAmount']):
    ax.hist(df[col], bins=25, color='#4C72B0', edgecolor='black')
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
colors = {'Yes': '#DD8452', 'No': '#4C72B0'}
for resp, group in df.groupby('CampaignResponse'):
    ax.scatter(group['AnnualIncome'], group['SpendingScore'],
               label=resp, alpha=0.6, s=25, color=colors[resp])
ax.set_xlabel("Annual Income")
ax.set_ylabel("Spending Score")
ax.set_title("Annual Income vs Spending Score, by Campaign Response")
ax.legend(title="Campaign Response")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
counts = df['CampaignResponse'].value_counts()
ax.bar(counts.index, counts.values, color=['#55A868', '#C44E52'])
ax.set_title("Campaign Response Counts")
ax.set_ylabel("Number of Customers")
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr = df[numerical_cols].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Heatmap")
plt.tight_layout()
plt.show()
print(corr.round(2))

**Three useful relationships from the heatmap:**
1. `PreviousPurchases` <-> `MembershipYears` (r=0.62) -- longer-tenured customers have naturally accumulated more purchases.
2. `SpendingScore` <-> `PurchaseAmount` (r=0.62) -- the spending score is a reasonably good proxy for actual dollars spent.
3. `PreviousPurchases` <-> `PurchaseAmount` (r=0.49) -- more frequent buyers tend to have higher total purchase amounts.

**5 observations from the plots:**
1. `SpendingScore` is heavily right-skewed with a hard ceiling at 100 -- the majority of customers (76.5%) score the maximum, so the histogram is not a natural bell curve.
2. `CampaignResponse` is extremely imbalanced (548 Yes vs 52 No) -- almost everyone responded to the campaign, which will matter for classification later.
3. In the Income vs Spending scatter, "No" responders (blue) are noticeably concentrated at lower spending scores -- customers who don't engage with spending also tend not to respond to campaigns.
4. `Age`, `AnnualIncome`, and `PurchaseAmount` all look roughly normally distributed, suggesting the dataset was likely generated (or sampled) without extreme real-world skew in these fields.
5. Correlations across most feature pairs are weak (under 0.3) -- the strongest relationships all involve purchase/tenure behavior, while demographic fields (`Age`, `Gender`) barely correlate with anything, hinting that behavior data will be more useful than demographics for segmentation and prediction.

## Part D: Customer Segmentation Using Clustering

In [ ]:
cluster_features = ['AnnualIncome', 'SpendingScore', 'WebsiteVisitsPerMonth',
                     'AvgSessionMinutes', 'PreviousPurchases', 'PurchaseAmount', 'MembershipYears']

scaler_c = StandardScaler()
X_cluster = scaler_c.fit_transform(df[cluster_features])

# Elbow method, k=1 to 10
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(K_range, inertias, marker='o')
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow Method for Optimal k")
ax.set_xticks(list(K_range))
plt.tight_layout()
plt.show()

print("Silhouette scores by k (2-8):")
sil_scores = {}
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_cluster)
    sil = silhouette_score(X_cluster, labels)
    sil_scores[k] = sil
    print(f"  k={k}: silhouette={sil:.4f}")
print(f"\nBest k by silhouette score alone: {max(sil_scores, key=sil_scores.get)}")

**Choosing k:** The elbow bends around k=3-4 (diminishing inertia drop after that point), and silhouette score peaks at k=3 (0.211). We select **k=4**: it's a close second on silhouette (0.161) while giving more business-actionable granularity than 3 broad groups -- a common, defensible trade-off between pure cluster cohesion and practical usefulness for targeting.

In [ ]:
CHOSEN_K = 4
kmeans_final = KMeans(n_clusters=CHOSEN_K, random_state=42, n_init=10)
kmeans_labels = kmeans_final.fit_predict(X_cluster)
kmeans_sil = silhouette_score(X_cluster, kmeans_labels)
print(f"K-Means (k={CHOSEN_K}) silhouette score: {kmeans_sil:.4f}")
print("Cluster sizes:", pd.Series(kmeans_labels).value_counts().sort_index().to_dict())

# Second clustering method: Agglomerative Clustering
agg = AgglomerativeClustering(n_clusters=CHOSEN_K)
agg_labels = agg.fit_predict(X_cluster)
agg_sil = silhouette_score(X_cluster, agg_labels)
print(f"\nAgglomerative (k={CHOSEN_K}) silhouette score: {agg_sil:.4f}")
print("Cluster sizes:", pd.Series(agg_labels).value_counts().sort_index().to_dict())

df['KMeans_Cluster'] = kmeans_labels
df['Agg_Cluster'] = agg_labels

In [ ]:
print("K-Means cluster profiles (mean values per cluster):")
profile = df.groupby('KMeans_Cluster')[cluster_features].mean().round(1)
display(profile)

print("\nCampaignResponse rate by cluster:")
print(df.groupby('KMeans_Cluster')['CampaignResponse'].apply(lambda x: (x == 'Yes').mean()).round(3))

print("\nHighValueCustomer rate by cluster:")
print(df.groupby('KMeans_Cluster')['HighValueCustomer'].apply(lambda x: (x == 'Yes').mean()).round(3))

**Comparing the two clustering methods:** K-Means achieved a slightly higher silhouette score (0.161) than Agglomerative Clustering (0.122) at k=4, suggesting K-Means found marginally more cohesive, well-separated groups on this data -- reasonable, since K-Means' centroid-based approach suits the roughly convex, similarly-sized clusters common in customer behavior data, while Agglomerative Clustering doesn't assume any particular cluster shape and can be more sensitive to how per-step merges compound.

**The 4 customer segments, described in simple words** (based on the K-Means profile):
- **Segment 2 -- Loyal high-value customers:** Longest membership (4.3 yrs), most website visits and previous purchases, highest purchase amounts. 86.6% are flagged high-value and 97% responded to the campaign -- clearly the company's best customers.
- **Segment 0 -- Premium spenders:** High income (~$84k), high spending score, but shorter tenure (1.8 yrs). Solid campaign response (91%) but only 38.5% high-value -- promising newer customers still building purchase history.
- **Segment 1 -- Value spenders:** Lower income (~$48k) but still a high spending score -- price-conscious customers who spend a large share of what they have. 42.4% high-value.
- **Segment 3 -- At-risk / low engagement:** Noticeably lower spending score (73 vs ~99 in other segments), lowest purchase amount, lowest campaign response (74.3%), and almost no high-value customers (1.4%) -- the segment most likely to be disengaging.

## Part E: PCA and t-SNE Visualization

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by 2 components:", pca.explained_variance_ratio_.sum().round(3))

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=df['KMeans_Cluster'], cmap='viridis', alpha=0.7, s=30)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)")
ax.set_title("Customer Segments Visualized with PCA")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.tight_layout()
plt.show()

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_cluster)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=df['KMeans_Cluster'], cmap='viridis', alpha=0.7, s=30)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
ax.set_title("Customer Segments Visualized with t-SNE")
legend1 = ax.legend(*scatter.legend_elements(), title="Cluster")
ax.add_artist(legend1)
plt.tight_layout()
plt.show()

**PCA vs t-SNE:** PCA's 2 components explain only ~53% of total variance, so a fair amount of structure is lost in the 2D projection -- clusters show real but somewhat blurred separation. t-SNE is purpose-built to preserve local neighborhood structure for visualization (not to preserve variance or global distances), so it typically produces visually tighter, more distinctly separated clusters at the cost of the axes having no direct interpretable meaning.

**Which is easier to interpret, and why:** t-SNE is usually easier to interpret *visually* -- the clusters look more clearly separated, making segment boundaries easy to eyeball. However, PCA is easier to interpret *analytically* -- its axes are actual directions in feature space with a quantifiable "% variance explained," so you can reason about what PC1/PC2 represent. For a business presentation, t-SNE is a great visual; for understanding which features drive separation, PCA is more useful.

## Part F: Supervised Learning

**Choosing the target:** We select **`HighValueCustomer`** over `CampaignResponse`. `CampaignResponse` is ~91% Yes / 9% No -- a trivial "always predict Yes" model would already score ~91% accuracy without learning anything meaningful, and with only 52 "No" examples in 600 rows, there's very little signal to train or evaluate on reliably. `HighValueCustomer` is well balanced (311 No / 289 Yes), giving a much fairer and more informative classification task.

In [ ]:
TARGET = "HighValueCustomer"
feature_cols_v1 = ['Age', 'AnnualIncome', 'SpendingScore', 'WebsiteVisitsPerMonth',
                    'AvgSessionMinutes', 'PreviousPurchases', 'MembershipYears', 'PurchaseAmount',
                    'Gender', 'DiscountUsed', 'EmailOpened']

df_model = df.copy()
for col in ['Gender', 'DiscountUsed', 'EmailOpened']:
    df_model[col] = LabelEncoder().fit_transform(df_model[col])

X_v1 = df_model[feature_cols_v1]
y = LabelEncoder().fit_transform(df_model[TARGET])  # No=0, Yes=1

# Quick check: how learnable is the target from a shallow tree? (leakage check)
dt_check = DecisionTreeClassifier(random_state=42, max_depth=3)
dt_check.fit(X_v1, y)
print("Feature importances (depth=3 tree, ALL features):")
for feat, imp in sorted(zip(feature_cols_v1, dt_check.feature_importances_), key=lambda x: -x[1]):
    print(f"  {feat}: {imp:.3f}")
print()
print(export_text(dt_check, feature_names=feature_cols_v1))

**Data leakage warning:** the depth-3 tree above splits *perfectly* using only `PurchaseAmount` and `PreviousPurchases` -- meaning `HighValueCustomer` is almost certainly a rule derived directly from those two columns when the dataset was created. Training a "predictive" model on them would just be reverse-engineering that rule, not learning genuinely predictive customer signals. We remove both features below for an honest evaluation.

In [ ]:
# Honest feature set -- excludes PurchaseAmount and PreviousPurchases (leakage sources)
feature_cols = ['Age', 'AnnualIncome', 'SpendingScore', 'WebsiteVisitsPerMonth',
                 'AvgSessionMinutes', 'MembershipYears', 'Gender', 'DiscountUsed', 'EmailOpened']

X = df_model[feature_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
scaler_f = StandardScaler()
X_train_scaled = scaler_f.fit_transform(X_train)
X_test_scaled = scaler_f.transform(X_test)

print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")
print(f"Class balance: {pd.Series(y).value_counts().to_dict()}")

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=6),
    "KNN": KNeighborsClassifier(n_neighbors=7),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=200),
}

results = {}
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, model) in zip(axes, models.items()):
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc, prec, rec, f1 = (accuracy_score(y_test, y_pred), precision_score(y_test, y_pred),
                            recall_score(y_test, y_pred), f1_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    results[name] = {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-score": f1}
    print(f"{name}: Acc={acc:.3f} Prec={prec:.3f} Rec={rec:.3f} F1={f1:.3f}")
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No", "Yes"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

comparison_df = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1-score"]].sort_values("F1-score", ascending=False)
print("\n===== Model Comparison (leakage-free) =====")
print(comparison_df.round(4))
print(f"\nBest model: {comparison_df.index[0]}")

In [ ]:
# Feature importance from Random Forest (which features drive HighValueCustomer, honestly)
rf_model = models["Random Forest"]
importances = sorted(zip(feature_cols, rf_model.feature_importances_), key=lambda x: -x[1])
print("Random Forest feature importances (leakage-free features):")
for feat, imp in importances:
    print(f"  {feat}: {imp:.3f}")

**Best model:** Logistic Regression, with the best F1-score (~0.76) on the honest feature set. It generalizes slightly better here than the tree-based models, which likely overfit on this dataset's relatively small size (600 rows, 480 training) once the leaking features are removed.

## Part G: Prompt Engineering and Business Recommendations

Five prompts explaining the customer segments and campaign response results, covering the five required types. Each entry includes the prompt itself, a response summary, a quality rating out of 5, and how the prompt was improved.

### 1. Basic prompt
**Prompt:** "Explain the customer segments in this dataset."

**Response summary:** A generic description listing that there are 4 clusters based on income, spending, and purchase behavior, without specifics, numbers, or actionable detail.

**Quality rating:** 2/5 -- technically correct but too vague to act on; doesn't reference actual cluster statistics or business implications.

**Improvement:** Added explicit statistics and a defined audience/purpose, leading to the data-driven and role-based prompts below.

---

### 2. Role-based prompt
**Prompt:** "You are a retail marketing strategist presenting customer segments to the VP of Marketing. Explain the 4 customer segments we found, in business language, focusing on what each segment means for revenue and retention."

**Response summary:** Produces business-framed segment names ("Loyal Champions," "Rising Spenders," "Budget-Conscious Loyalists," "At-Risk / Disengaging") with revenue and retention implications for each, in language appropriate for a non-technical executive audience.

**Quality rating:** 4/5 -- good tone and framing, but occasionally invents plausible-sounding numbers not grounded in the actual cluster statistics.

**Improvement:** Combined with the data-driven prompt below to force grounding in real numbers.

---

### 3. Step-by-step prompt
**Prompt:** "Walk through, step by step: (1) how the 4 clusters were formed, (2) what features drove the separation, (3) how each cluster differs from the others, (4) what business action fits each cluster."

**Response summary:** Produces a structured, numbered walkthrough covering the clustering method (K-Means on 7 scaled behavior features), the top differentiating features (SpendingScore, PurchaseAmount, MembershipYears), a compare/contrast of all 4 clusters, and one action recommendation per cluster.

**Quality rating:** 5/5 -- the explicit structure prevents the vagueness of the basic prompt and produces a genuinely useful, complete artifact.

**Improvement:** This was already strong; minor tweak was adding "compare/contrast" to step 3 to force explicit differentiation rather than 4 separate descriptions.

---

### 4. Data-driven prompt using clustering statistics
**Prompt:** "Given these K-Means cluster statistics -- Cluster 0: income $84,325, spending score 99.3, tenure 1.8 yrs, high-value rate 38.5%; Cluster 1: income $48,542, spending score 98.2, tenure 1.7 yrs, high-value rate 42.4%; Cluster 2: income $71,137, spending score 99.8, tenure 4.3 yrs, high-value rate 86.6%; Cluster 3: income $58,041, spending score 73.0, tenure 1.5 yrs, high-value rate 1.4% -- explain what distinguishes each segment and which one is most valuable to the business."

**Response summary:** Correctly identifies Cluster 2 as the standout high-value segment (long tenure + near-universal high-value rate), and Cluster 3 as clearly the weakest (much lower spending score, near-zero high-value rate) despite similar income to Cluster 1 -- directly grounded in the actual numbers rather than generic descriptions.

**Quality rating:** 5/5 -- most accurate and specific of all five, precisely because it's anchored to real statistics rather than left to infer or invent them.

**Improvement:** This is the most improved version of the basic prompt -- feeding in real numbers eliminated the vagueness and invention problems seen in prompts 1 and 2.

---

### 5. Business recommendation prompt
**Prompt:** "Based on the 4 customer segments and their high-value/campaign-response rates, recommend which segment the marketing team should target first for the next campaign, and suggest one concrete action for each of the other three segments."

**Response summary:** Recommends prioritizing Cluster 3 (at-risk/low engagement) for a re-engagement campaign given its low response and high-value rates, since it represents the biggest gap versus its similarly-sized peers; suggests loyalty rewards for Cluster 2 (retain the best customers), upsell/cross-sell offers for Cluster 0 (high income, still building tenure), and value-oriented bundle deals for Cluster 1 (price-conscious but engaged).

**Quality rating:** 4/5 -- actionable and well-reasoned, though it would benefit from suggested KPIs to track success (added as a follow-up refinement).

**Improvement:** Iterated to explicitly request "one concrete action for each segment" rather than a general summary, which forced specific, differentiated recommendations instead of one-size-fits-all advice.

## Questions

In [ ]:
# Q35 data: Do customers who opened emails have a higher campaign response rate?
print("CampaignResponse rate by EmailOpened:")
print(df.groupby('EmailOpened')['CampaignResponse'].apply(lambda x: (x == 'Yes').mean()).round(3))

In [ ]:
# Q36 data: Are high-income customers always high-spending?
print("Correlation (AnnualIncome, SpendingScore):", df['AnnualIncome'].corr(df['SpendingScore']).round(3))

income_median = df['AnnualIncome'].median()
spending_median = df['SpendingScore'].median()
df['IncomeTier'] = np.where(df['AnnualIncome'] >= income_median, 'High Income', 'Low Income')
df['SpendingTier'] = np.where(df['SpendingScore'] >= spending_median, 'High Spending', 'Low Spending')
print(pd.crosstab(df['IncomeTier'], df['SpendingTier'], normalize='index').round(3))

**34. Which features are most useful for identifying high-value customers?**
Using the honest (leakage-free) Random Forest importances: `MembershipYears` (0.20), `AvgSessionMinutes` (0.17), and `AnnualIncome` (0.17) are the top drivers -- tenure and engagement matter more than raw demographics. (Note: `PurchaseAmount` and `PreviousPurchases` are the *strongest* predictors overall, but were excluded from this ranking since the label itself appears to be derived from them -- see Part F's leakage discussion.)

**35. Do customers who opened emails have a higher campaign response rate?**
Yes. Customers who opened emails responded to the campaign 94.4% of the time, versus 85.3% for those who didn't -- roughly a 9-point gap, suggesting email engagement is a real (if not huge) signal of campaign receptiveness.

**36. Are high-income customers always high-spending customers? Support your answer using plots.**
No. The Income vs Spending scatter plot (Part C) and the correlation coefficient (r=0.195, quite weak) both show income and spending are only loosely related. The crosstab confirms this concretely: 12.5% of high-income customers are actually *low* spenders, while a substantial 65.3% of low-income customers are still high spenders -- income clearly doesn't determine spending behavior on its own.

**37. Which customer segment should the company target first, and why?**
**Segment 3 (at-risk/low engagement)** should be the first target. It has income comparable to Segment 1, yet a dramatically lower spending score (73 vs ~99) and campaign response rate (74.3% vs 91%+ elsewhere), and almost no high-value customers (1.4%). This is the segment with the most "headroom" to convert -- customers who have spending capacity but aren't currently engaging, making them a natural re-engagement campaign target. (Segment 2, the loyal high-value group, is also strategically important, but for *retention* rather than acquisition/conversion effort.)

**38. What are the limitations of your model and clustering result?**
- **Data leakage in the classification target:** `HighValueCustomer` is near-perfectly derivable from `PurchaseAmount` and `PreviousPurchases` alone (a depth-3 tree gets 100% accuracy using just those two columns), strongly suggesting the label was rule-derived from them during dataset creation. Any model trained on those features doesn't demonstrate real predictive power -- we address this by excluding them, but the resulting ~72-76% accuracy is a more honest (if less impressive) estimate.
- **Small, possibly synthetic dataset:** 600 rows is a small sample for both clustering and classification; the very regular feature distributions (and the apparent rule-based target) suggest this may be synthetic data, which may not fully reflect real customer behavior patterns or edge cases.
- **Class imbalance in CampaignResponse:** with only 52 "No" examples, that target was not usable for reliable classification -- any model built on it would need to be evaluated very carefully (e.g. via precision/recall on the minority class, not accuracy).
- **Moderate clustering cohesion:** silhouette scores across all tested k values stayed under 0.22, indicating the clusters are real but have meaningfully overlapping boundaries rather than being cleanly separated -- some customers near cluster edges could reasonably belong to more than one segment.
- **PCA's limited variance capture:** the 2D PCA projection explains only ~53% of total variance, so the visual cluster separation understates some of the real underlying structure in the full 7-feature space.